# Generate Real Sentence Embeddings for PyBot's RAG Knowledge Base

Run this notebook in **Google Colab** (GPU runtime optional but recommended) to upgrade PyBot's retriever from TF-IDF to real semantic embeddings.

**Steps:**
1. Upload `knowledge_base.json` (from `college_chatbot/data/`) when prompted below.
2. Run all cells.
3. Download the generated `kb_embeddings.npy` file.
4. Place it inside your project's `college_chatbot/model/` folder.
5. Restart `app.py` — it will automatically detect and use the dense embeddings instead of TF-IDF.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import files
import json

print('Please upload knowledge_base.json')
uploaded = files.upload()
kb_filename = list(uploaded.keys())[0]

with open(kb_filename, 'r', encoding='utf-8') as f:
    kb = json.load(f)

print(f'Loaded {len(kb)} chunks')

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# all-MiniLM-L6-v2 is small, fast, and a strong general-purpose choice for RAG
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [f"{c['title']}. {c['text']}" for c in kb]
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

print('Embeddings shape:', embeddings.shape)

In [ ]:
np.save('kb_embeddings.npy', embeddings)
files.download('kb_embeddings.npy')
print('Done! Move kb_embeddings.npy into college_chatbot/model/ in your project.')

## Optional: also embed queries the same way at inference time

If you want the Flask app itself to compute *query* embeddings with the same model (instead of only having pre-embedded knowledge-base chunks), install `sentence-transformers` locally too and update `app.py`'s `embed_query()` call to use:

```python
from sentence_transformers import SentenceTransformer
_embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_query(text):
    return _embed_model.encode([text], normalize_embeddings=True)[0]
```

This requires `torch` + `sentence-transformers` on the machine running `app.py`. If that's too heavy for your laptop, keep the default TF-IDF retriever — it works well for a keyword-rich FAQ/course-notes knowledge base like this one.